# Finding Z analysis
Run the cells from the top. This is your editable copy; later template updates do not change it. Saved settings reproduce the web analysis using the original available samples, not a copy of event data. Deleted or changed samples affect reproducibility.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from findingz.catalog import load_catalog
from findingz.delphes import default_run_root, open_run
from findingz.hypotheses import build_sample_library, validate_counting_samples
from findingz.analysis_variables import AnalysisVariable, apply_windows
from findingz.counting import summarize_cut_and_count
from dataclasses import asdict

## Analysis settings
Finding Z replaces only the tagged cell below. Edit these settings to explore. Blank templates begin with no selected samples. Plot and counting cuts are independent.

In [ ]:
analysis = {"plot": None, "count": None}

In [ ]:
library = build_sample_library(load_catalog(), default_run_root())
display(pd.DataFrame([{"sample_id": s.sample_id, "label": s.label, "configuration": s.config,
                      "cross_section_pb": s.cross_section_pb} for s in library.values()]))
# For a blank analysis, use IDs from this table to fill analysis["plot"] or analysis["count"].
# Plot example:
# analysis["plot"] = {"samples": ["YOUR_SAMPLE_ID"], "observable": "mll",
#     "variables": {"mll": {"column": "mll", "label": "Dilepton mass [GeV]",
#                          "description": "Invariant mass of the lepton pair", "step": 1}},
#     "channels": ["ee", "mumu"], "windows": {}, "expected_yields": False, "luminosity_fb": 1}


## Event selection
These cuts act on the same flat event variables as the web interface.

In [ ]:
def selected_frames(settings, ids):
    variables = {key: AnalysisVariable(**value) for key, value in settings["variables"].items()}
    frames = {}
    for sample_id in ids:
        if sample_id not in library:
            raise ValueError(f"Sample {sample_id} is no longer available. Restore it or choose another ID.")
        sample = library[sample_id]
        frame = sample.expected_frame(settings["luminosity_fb"]) if settings.get("expected_yields", True) else sample.load()
        if "channel" in frame:
            frame = frame.loc[frame["channel"].isin(settings["channels"])]
        frames[sample_id] = apply_windows(frame, settings["windows"], variables)
    return frames, variables

## Plot distributions

In [ ]:
settings = analysis["plot"]
if settings:
    frames, variables = selected_frames(settings, settings["samples"])
    variable = variables[settings["observable"]]
    series = [frame[variable.column] for frame in frames.values() if not frame.empty]
    values = pd.to_numeric(pd.concat(series, ignore_index=True), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna() if series else pd.Series(dtype=float)
    low, high = (float(values.min()), float(values.max())) if not values.empty else (0., 1.)
    if low == high:
        padding = max(abs(low) * .05, .5)
        low, high = low - padding, high + padding
    bins = np.linspace(low, high, 41)
    fig, ax = plt.subplots(figsize=(8, 5))
    for sample_id, frame in frames.items():
        weights = pd.to_numeric(frame.get("weight", pd.Series(1., index=frame.index)))
        if not settings["expected_yields"] and weights.sum() > 0:
            weights = weights / weights.sum()
        ax.hist(frame[variable.column], bins=bins, weights=weights, histtype="step", label=library[sample_id].label)
    ax.set_xlabel(variable.label)
    ax.set_ylabel("Expected events" if settings["expected_yields"] else "Fraction of sample")
    ax.grid(alpha=.2)
    ax.legend()
    plt.show()
else:
    print("Choose plot settings above to begin.")

## Cut-and-count
Expected limits are teaching-level Gaussian approximations with a three-event floor, not CLs or a likelihood fit. This cell recomputes the estimate; it does not export observed data.

In [ ]:
settings = analysis["count"]
if settings:
    ids = [settings["signal"], *settings["backgrounds"]]
    validate_counting_samples([library[key] for key in ids])
    frames, _ = selected_frames(settings, ids)
    result = summarize_cut_and_count(frames[settings["signal"]],
        pd.concat([frames[key] for key in settings["backgrounds"]], ignore_index=True),
        background_uncertainty_fraction=settings["background_uncertainty_fraction"])
    display(pd.Series(asdict(result)))
    cross_section = library[settings["signal"]].cross_section_pb
    if cross_section is not None:
        print("Approximate cross-section limit [pb]:", result.approximate_expected_signal_strength_limit * cross_section)
else:
    print("No complete counting selection was saved. Define signal/backgrounds, variables, channels, windows, luminosity_fb and background_uncertainty_fraction to continue.")

## Optional: what is an electron?
For a full-pipeline run with retained Delphes ROOT, uncomment this cell and choose a run ID. These object-level changes are exploratory: they do not silently replace the flat CSV variables or cuts above. Reconstruct your observables from these selected objects to study their effect.

In [ ]:
# events = open_run("YOUR_RUN_ID")
# electrons = events.electrons
electron_pt_min = 10.0  # GeV: edit these object definitions
electron_eta_max = 2.5
# selected_electrons = electrons[(electrons.pt > electron_pt_min) & (abs(electrons.eta) < electron_eta_max)]
# Inspect events.available_collections for other retained detector objects.